# Fine-Tuning a Small Transformer for Sentiment Classification

This notebook demonstrates the main stages of **Transformer fine-tuning** using a deliberately small text-classification example.

It is designed as teaching material and is **not part of the Arabic machine-translation research experiments**.

## Learning objectives

By the end of this lesson, students should be able to:

1. Explain the difference between pretraining and fine-tuning.
2. Tokenize text for a pretrained Transformer.
3. Prepare batches using a PyTorch `Dataset` and `DataLoader`.
4. Fine-tune a Transformer using a standard training loop.
5. Evaluate the model and use it for new predictions.

> **Scope:** This notebook fine-tunes a small pretrained BERT model. Training a Transformer from random initialization requires substantially more data and computation.

## 2. Import libraries and set reproducibility controls

A fixed random seed makes the data shuffling and initial classification-head weights more consistent across runs.

In [ ]:
import random
from typing import Dict, List, Tuple

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
import torch
from torch.utils.data import DataLoader, Dataset

from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
)

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Create a small teaching dataset

The dataset is included directly in the notebook so that students can focus on the training workflow rather than file handling.

- Label `0`: negative sentiment
- Label `1`: positive sentiment

This dataset is intentionally small. Its purpose is to demonstrate the pipeline, not to produce a production-quality sentiment classifier.

In [ ]:
train_examples: List[Tuple[str, int]] = [
    ("I enjoyed the lecture and learned a lot.", 1),
    ("The explanation was clear and helpful.", 1),
    ("This course is interesting and well organized.", 1),
    ("The example made the concept easy to understand.", 1),
    ("I am satisfied with the lesson.", 1),
    ("The activity was useful and engaging.", 1),
    ("The instructor explained the topic very well.", 1),
    ("The notebook was simple and informative.", 1),
    ("I liked the practical demonstration.", 1),
    ("The material was excellent.", 1),
    ("This was a valuable learning experience.", 1),
    ("The exercise helped me understand the model.", 1),
    ("The lecture was confusing and difficult to follow.", 0),
    ("The explanation was unclear.", 0),
    ("This course is poorly organized.", 0),
    ("The example did not help me understand the concept.", 0),
    ("I am disappointed with the lesson.", 0),
    ("The activity was repetitive and unhelpful.", 0),
    ("The instructor moved too quickly.", 0),
    ("The notebook was complicated and incomplete.", 0),
    ("I did not like the practical demonstration.", 0),
    ("The material was frustrating.", 0),
    ("This was not a useful learning experience.", 0),
    ("The exercise made the topic more confusing.", 0),
]

test_examples: List[Tuple[str, int]] = [
    ("The lesson was clear and enjoyable.", 1),
    ("I found the notebook useful.", 1),
    ("The demonstration was easy to follow.", 1),
    ("The course content was well presented.", 1),
    ("The lesson was difficult and unclear.", 0),
    ("I found the notebook unhelpful.", 0),
    ("The demonstration was confusing.", 0),
    ("The course content was badly presented.", 0),
]

print(f"Training examples: {len(train_examples)}")
print(f"Test examples: {len(test_examples)}")

## 4. Load a tokenizer and pretrained Transformer

We use `prajjwal1/bert-tiny`, a compact BERT configuration with two Transformer layers. It is suitable for demonstrating fine-tuning on modest hardware.

`AutoTokenizer` converts text into:

- `input_ids`: vocabulary indices;
- `attention_mask`: indicators showing which positions contain real tokens rather than padding.

`AutoModelForSequenceClassification` adds a classification head on top of the pretrained Transformer.

In [ ]:
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

id2label = {
    0: "NEGATIVE",
    1: "POSITIVE",
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1,
}

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    problem_type="single_label_classification",
)
model.to(device)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Model: {MODEL_NAME}")
print(f"Total parameters: {parameter_count:,}")
print(f"Trainable parameters: {trainable_count:,}")

## 5. Inspect tokenization

Special tokens are added automatically:

- `[CLS]` summarizes the sequence for classification;
- `[SEP]` marks the end of the sequence;
- `[PAD]` fills unused positions in a fixed-length batch.

In [ ]:
sample_text = "The lesson was clear and useful."

sample_encoding = tokenizer(
    sample_text,
    truncation=True,
    padding="max_length",
    max_length=24,
)

sample_tokens = tokenizer.convert_ids_to_tokens(sample_encoding["input_ids"])

print("Text:")
print(sample_text)
print("\nTokens:")
print(sample_tokens)
print("\nAttention mask:")
print(sample_encoding["attention_mask"])

## 6. Build a PyTorch dataset

Each item contains tokenized model inputs and its correct label.

In [ ]:
class SentimentDataset(Dataset):
    def __init__(
        self,
        examples: List[Tuple[str, int]],
        tokenizer,
        max_length: int = 48,
    ) -> None:
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        text, label = self.examples[index]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


train_dataset = SentimentDataset(train_examples, tokenizer)
test_dataset = SentimentDataset(test_examples, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
)

first_batch = next(iter(train_loader))

for name, tensor in first_batch.items():
    print(f"{name}: {tuple(tensor.shape)}")

## 7. Define the training and evaluation functions

For every training batch, the program:

1. clears old gradients;
2. performs a forward pass;
3. calculates cross-entropy loss;
4. propagates gradients backward;
5. updates the model parameters.

During evaluation, gradients are disabled because no parameter update is required.

In [ ]:
def move_batch_to_device(
    batch: Dict[str, torch.Tensor],
    target_device: torch.device,
) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.to(target_device)
        for name, tensor in batch.items()
    }


def train_one_epoch(
    model,
    data_loader: DataLoader,
    optimizer: AdamW,
    target_device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0

    for batch in data_loader:
        batch = move_batch_to_device(batch, target_device)

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


def evaluate(
    model,
    data_loader: DataLoader,
    target_device: torch.device,
) -> Tuple[float, float]:
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    example_count = 0

    with torch.no_grad():
        for batch in data_loader:
            batch = move_batch_to_device(batch, target_device)

            outputs = model(**batch)
            predictions = outputs.logits.argmax(dim=-1)

            total_loss += outputs.loss.item()
            correct_predictions += (
                predictions == batch["labels"]
            ).sum().item()
            example_count += batch["labels"].size(0)

    average_loss = total_loss / len(data_loader)
    accuracy = correct_predictions / example_count

    return average_loss, accuracy

## 8. Fine-tune the model

Because the dataset is tiny, several epochs are used to make the training behavior visible. In a real project, the number of epochs and learning rate should be selected using a validation set rather than the test set.

In [ ]:
LEARNING_RATE = 3e-5
EPOCHS = 5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device,
    )

    test_loss, test_accuracy = evaluate(
        model,
        test_loader,
        device,
    )

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "test_loss": test_loss,
            "test_accuracy": test_accuracy,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"test loss: {test_loss:.4f} | "
        f"test accuracy: {test_accuracy:.2%}"
    )

## 9. Examine the test predictions

Accuracy on eight simple examples is not a reliable scientific evaluation. This section only demonstrates how predictions are generated and compared with labels.

In [ ]:
model.eval()

with torch.no_grad():
    for text, true_label in test_examples:
        encoded = tokenizer(
            text,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )
        encoded = {
            name: tensor.to(device)
            for name, tensor in encoded.items()
        }

        logits = model(**encoded).logits
        predicted_label = logits.argmax(dim=-1).item()

        print(f"Text: {text}")
        print(f"True: {id2label[true_label]}")
        print(f"Predicted: {id2label[predicted_label]}")
        print("-" * 60)

## 10. Create a reusable prediction function

Softmax converts the model's raw logits into class probabilities. These probabilities reflect the model's relative confidence, but they should not automatically be interpreted as calibrated probabilities.

In [ ]:
def predict_sentiment(text: str) -> Dict[str, float | str]:
    model.eval()

    encoded = tokenizer(
        text,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )
    encoded = {
        name: tensor.to(device)
        for name, tensor in encoded.items()
    }

    with torch.no_grad():
        logits = model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1).squeeze(0)

    predicted_id = probabilities.argmax().item()

    return {
        "label": id2label[predicted_id],
        "confidence": float(probabilities[predicted_id]),
    }


examples = [
    "The workshop was informative and enjoyable.",
    "The explanation was difficult to understand.",
]

for example in examples:
    print(example)
    print(predict_sentiment(example))
    print()

## 11. Optional: save the fine-tuned model

Saving both the model and tokenizer allows the same preprocessing and learned weights to be loaded later.

In [ ]:
SAVE_DIRECTORY = "outputs/bert_tiny_sentiment"

model.save_pretrained(SAVE_DIRECTORY)
tokenizer.save_pretrained(SAVE_DIRECTORY)

print(f"Saved model and tokenizer to: {SAVE_DIRECTORY}")

## 12. Discussion

### What was pretrained?

The BERT encoder had already learned general language representations from a large corpus.

### What was fine-tuned?

All model parameters, including the classification head, were updated using the small sentiment dataset.

### Why use a small model?

A small model reduces memory use and training time, allowing students to observe the full workflow on CPU or a modest GPU.

### Limitations

- The dataset is synthetic and very small.
- The test set is too small for a reliable performance claim.
- The examples contain simple vocabulary and obvious sentiment cues.
- The lesson does not demonstrate hyperparameter search, cross-validation, class imbalance, or robust error analysis.
- The model and tokenizer are English-only.

## Exercises

1. Add neutral examples and convert the task to three-class classification.
2. Increase the dataset size and create separate training, validation, and test sets.
3. Freeze the BERT encoder and train only the classification head.
4. Compare the training time and accuracy before and after freezing.
5. Replace the model with a multilingual Transformer and add Arabic examples.
6. Plot training and validation loss across epochs.
7. Examine misclassified examples and propose explanations.

## References

- Hugging Face Transformers — Text classification:  
  https://huggingface.co/docs/transformers/main/tasks/sequence_classification
- Hugging Face Transformers — Trainer and training concepts:  
  https://huggingface.co/docs/transformers/trainer
- BERT Tiny model card:  
  https://huggingface.co/prajjwal1/bert-tiny